In [3]:
import pandas as pd

df = pd.read_csv('superstore.csv', encoding='windows-1252')

In [ ]:
def row_to_text(row):
    '''
    Add column to dataframe with natural language description
    '''
    description = (
        f"On {row['Order Date']}, {row['Customer Name']} from {row['City']}, {row['State']} in the {row['Region']} region"
        f"purchased {row['Quantity']} unit(s) of '{row['Product Name']}' "
        f"for a total of ${row['Sales']:.2f}, generating a profit of ${row['Profit']:.2f} with a {row['Discount']} discount."
        f"The category of the purchased product is {row['Category']} and {row['Sub-Category']} sub-category"
    )
    return description

df['natural_language'] = df.apply(row_to_text, axis=1)

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,natural_language,year_month
0,1,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,"On 2016-11-08 00:00:00, Claire Gute from Hende...",2016-11
1,2,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,"On 2016-11-08 00:00:00, Claire Gute from Hende...",2016-11
2,3,CA-2016-138688,2016-06-12,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,"On 2016-06-12 00:00:00, Darrin Van Huff from L...",2016-06
3,4,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,"On 2015-10-11 00:00:00, Sean O'Donnell from Fo...",2015-10
4,5,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,"On 2015-10-11 00:00:00, Sean O'Donnell from Fo...",2015-10


In [5]:
'''
Create summaries of main categories
'''

def create_monthly_summaries(df): #Asked google gemini how to group by month
    df['Order Date'] = pd.to_datetime(df['Order Date'])
    df['year_month'] = df['Order Date'].dt.to_period('M')
    monthly_summary = df.groupby('year_month')[['Sales','Profit']].sum()

    return monthly_summary

def create_category_summary(df):
    category_summary = df.groupby('Category')[['Sales', 'Profit']].sum()
    return category_summary

def create_region_summary(df):
    region_summary = df.groupby('State')[['Sales','Profit']].sum()
    return region_summary

summary_m = create_monthly_summaries(df)

summary_cat = create_category_summary(df)

summary_reg = create_region_summary(df)
summary_reg.head()


,Sales,Profit
State,,
Alabama,19510.6400,5786.8253
Arizona,35282.0010,-3427.9246
Arkansas,11678.1300,4008.6871
California,457687.6315,76381.3871
Colorado,32108.1180,-6527.8579


In [6]:
def generate_stats(df):
    report_lines = []
    report_lines.append("Overall stats")
    stats = df[['Sales', 'Quantity', 'Profit']].describe().round(2)

    for col in stats.columns:
        report_lines.append(f"\nMetric: {col}")
        report_lines.append(f"  - Total Count: {stats.loc['count', col]}")
        report_lines.append(f"  - Average (Mean): {stats.loc['mean', col]}")
        report_lines.append(f"  - Standard Deviation: {stats.loc['std', col]}")
        report_lines.append(f"  - Minimum: {stats.loc['min', col]}")
        report_lines.append(f"  - Median (50th percentile): {stats.loc['50%', col]}")
        report_lines.append(f"  - Maximum: {stats.loc['max', col]}")


    report_lines.append("\n Category stats\n")

    cat_stats = df.groupby('Category')[['Sales', 'Profit']].mean().round(2)

    for cat, row in cat_stats.iterrows():
        report_lines.append(f"- {cat}: Average Sales of ${row['Sales']} with Average Profit of ${row['Profit']}.")

    report_lines.append("\n Region stats\n")

    reg_stats = df.groupby('State')[['Sales', 'Profit']].mean().round(2)

    for reg, row in reg_stats.iterrows():
        report_lines.append(f"- {reg}: Average Sales of ${row['Sales']} with Average Profit of ${row['Profit']}.")

    final_report_text = "\n".join(report_lines)

    file_path = "statistical_summary_report.txt"
    with open(file_path, "w") as file:
        file.write(final_report_text)

generate_stats(df)

In [29]:
def chunk_text(text, chunk_size, overlap=0):
    """
    Splits a given text into chunks of a specific character size
    """       
    chunks = []
    step_size = chunk_size - overlap
    
    for i in range(0, len(text), step_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
        
    return chunks

In [ ]:
def chunk_dataframe_content(df, chunk_size, overlap_pct=0.10):
    """
    Chunks the natural language column of the dataframe.
    """
    full_text = "\n".join(df['natural_language'].tolist())
    
    overlap = int(chunk_size * overlap_pct)
    return chunk_text(full_text, chunk_size, overlap)

chunk_sizes = [500, 1000, 2000]
results = {}

for size in chunk_sizes:
    chunks = chunk_dataframe_content(df, chunk_size=size)
    results[size] = chunks
    print(f"Size {size}: Created {len(chunks)} chunks.")

print(results[1000][0])

''' 
Going for 1000 character chunk size to get around 4-5 transactions for each chunk
'''

Size 500: Created 6636 chunks.
Size 1000: Created 3318 chunks.
Size 2000: Created 1659 chunks.
On 2016-11-08 00:00:00, Claire Gute from Henderson, Kentucky in the South regionpurchased 2 unit(s) of 'Bush Somerset Collection Bookcase' for a total of $261.96, generating a profit of $41.91 with a 0.0 discount.The category of the purchased product is Furniture and Bookcases sub-category
On 2016-11-08 00:00:00, Claire Gute from Henderson, Kentucky in the South regionpurchased 3 unit(s) of 'Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back' for a total of $731.94, generating a profit of $219.58 with a 0.0 discount.The category of the purchased product is Furniture and Chairs sub-category
On 2016-06-12 00:00:00, Darrin Van Huff from Los Angeles, California in the West regionpurchased 2 unit(s) of 'Self-Adhesive Address Labels for Typewriters by Universal' for a total of $14.62, generating a profit of $6.87 with a 0.0 discount.The category of the purchased product is Office Supplies 

' \nGoing for 1000 character chunk size to get around 6 transactions for each chunk\n'

In [31]:
import chromadb as cdb
from chromadb.utils import embedding_functions
client = cdb.PersistentClient(path='./vector_db')

embedding = embedding_functions.DefaultEmbeddingFunction()

collection = client.get_or_create_collection(name='my_collection', embedding_function=embedding)

In [32]:
def chunks_to_db(chunks, collection):
    """
    Generates embeddings and stores chunks in ChromaDB.
    """
    ids = [f"id_{i}" for i in range(len(chunks))]
    
    collection.add(
        documents=chunks,
        ids=ids
    )
    print(f"Loaded {len(chunks)} chunks into database.")

chunks_to_db(results[1000], collection)

Loaded 3318 chunks into database.


In [33]:
def prepare_chunks_with_metadata(df, rows_per_chunk=5):
    """
    Groups rows into chunks and aggregates metadata for each group.
    """
    documents = []
    metadatas = []
    ids = []
    
    for i in range(0, len(df), rows_per_chunk):
        subset = df.iloc[i : i + rows_per_chunk]
        
        chunk_text = "\n".join(subset['natural_language'].tolist())
        
        # Used google gemini to aggregate metadata for the chunk
        metadata = {
            "start_date": subset['Order Date'].min().strftime('%Y-%m-%d'),
            "end_date": subset['Order Date'].max().strftime('%Y-%m-%d'),
            "categories": ", ".join(subset['Category'].unique()),
            "regions": ", ".join(subset['Region'].unique()),
            "states": ", ".join(subset['State'].unique()),
            "total_sales": float(subset['Sales'].sum())
        }
        
        documents.append(chunk_text)
        metadatas.append(metadata)
        ids.append(f"chunk_{i}")
        
    return documents, metadatas, ids

docs, meta, ids = prepare_chunks_with_metadata(df)
collection.add(documents=docs, metadatas=meta, ids=ids)

In [ ]:
def retrieve_data(query, n_results=3, category_filter=None, state_filter=None, region_filter=None, start_date=None, end_date=None):
    """
    Retrieves chunks based on semantic similarity and optional metadata filters.
    """
    where_clause = None
    
    if category_filter or state_filter:
        conditions = []
        if category_filter:
            conditions.append({"categories": {"$contains": category_filter}})
        if state_filter:
            conditions.append({"states": {"$eq": state_filter}})
        if region_filter:
            conditions.append({"regions": {"$like": region_filter}})
        if start_date:
            conditions.append({"start_date": {"$gte": start_date}})
        if end_date:
            conditions.append({"end_date": {"$lte": end_date}})
        
        if len(conditions) > 1:
            where_clause = {"$and": conditions}
        elif len(conditions) == 1:
            where_clause = conditions[0]

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_clause
    )
    
    return results

res1 = retrieve_data("high value furniture sales")

res2 = retrieve_data("high value", state_filter="Florida")
print(res1)
print(res2)

{'ids': [['id_2769', 'id_2066', 'id_2822']], 'embeddings': None, 'documents': [["enerating a profit of $6.59 with a 0.2 discount.The category of the purchased product is Furniture and Furnishings sub-category\nOn 2014-03-21 00:00:00, Gary McGarr from Knoxville, Tennessee in the South regionpurchased 3 unit(s) of 'Global Highback Leather Tilter in Burgundy' for a total of $218.38, generating a profit of $-10.92 with a 0.2 discount.The category of the purchased product is Furniture and Chairs sub-category\nOn 2015-04-30 00:00:00, Jane Waco from Lawrence, Massachusetts in the East regionpurchased 5 unit(s) of 'Eldon 200 Class Desk Accessories, Burgundy' for a total of $31.40, generating a profit of $13.19 with a 0.0 discount.The category of the purchased product is Furniture and Furnishings sub-category\nOn 2015-04-30 00:00:00, Jane Waco from Lawrence, Massachusetts in the East regionpurchased 1 unit(s) of 'DAX Two-Tone Rosewood/Black Document Frame, Desktop, 5 x 7' for a total of $9.48, 

In [41]:
# Llama 3.2 3B:  ollama pull llama3.2:3b
import ollama

def generate_rag_response(query, n_results=3, category_filter=None, state_filter=None, region_filter=None, start_date=None, end_date=None):
    """
    RAG Pipeline: Retrieves context from ChromaDB and generates an answer using Llama 3.2.
    google gemini was used to generate prompt engineering quidelines and tests

    Gemini used to handle bugfixes
    """
    results = retrieve_data(query, n_results=n_results, category_filter=category_filter, state_filter=state_filter, region_filter=region_filter, start_date=start_date, end_date=end_date)
    
    if not results['documents'] or not results['documents'][0]:
        return "No relevant transactions in the database for that query."
        
    retrieved_chunks = results['documents'][0]
    
    context_block = "\n\n---\n\n".join(retrieved_chunks)
    
    system_prompt = f"""You are an expert E-commerce Data Analyst. 
    Your job is to answer the user's question based strictly on the provided transaction records.
    
    Rules:
    1. If the answer cannot be found in the provided context, say "I don't have enough data to answer that."
    2. Do not hallucinate or make up sales data.
    3. Be concise and analytical in your response.

    CONTEXT (Recent Transactions):
    {context_block}
    """
    
    print("Generating response (Llama 3.2)...\n")
    
    try:
        response = ollama.chat(
            model='llama3.2:3b', 
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': query}
            ]
        )
        return response['message']['content']
        
    except Exception as e:
        return f"Error connecting to Ollama: {e}"


In [42]:
# Sample queries generated by Google Gemini

# Test 1: Standard Query
query_1 = "What furniture products did Parhena Norris buy, and were they profitable?"
answer_1 = generate_rag_response(query_1)
print(f"Q: {query_1}\nAnswer: {answer_1}\n")

# Test 2: Filtered Query
query_2 = "What were the most recent high value sales?"
answer_2 = generate_rag_response(query_2, state_filter="Florida", category_filter="Furniture")
print(f"Q: {query_2} (Filtered to Florida Furniture)\nAnswer: {answer_2}\n")

Generating response (Llama 3.2)...

Q: What furniture products did Parhena Norris buy, and were they profitable?
Answer: Based on the provided transaction records, Parhena Norris purchased the following furniture products:

1. 'O'Sullivan 4-Shelf Bookcase in Odessa Pine' - The total purchase price was $387.14, resulting in a profit of -$14.52 (not profitable)
2. 'Tenex Chairmats For Use with Hard Floors' - The total purchase price was $77.95, resulting in a profit of -$11.69 (not profitable)
3. 'Avery 496' - The total purchase price was $3.00, resulting in a profit of $1.05 (profitable)

No other furniture products were purchased by Parhena Norris.

Q: What were the most recent high value sales? (Filtered to Florida Furniture)
Answer: No relevant transactions in the database for that query.



In [36]:
query_3 = "Which product category generates the most revenue?"
answer_3 = generate_rag_response(query_3)
print(f"Q: {query_3}\nAnswer: {answer_3}")

Generating response (Llama 3.2)...

Q: Which product category generates the most revenue?
Answer: Based on the provided transaction records, I can analyze the data to determine which product category generates the most revenue.

From the transactions, we have three categories that seem to be well-represented: Office Supplies and Labels, Technology and Accessories, and Technology and Phones. However, it's worth noting that there is a single record for "Furniture and Tables" but only two records for this category, whereas all other categories have more extensive data sets.

Given the information available:

- Office Supplies and Labels sub-category generated revenue from one transaction (7 units of 'Alphabetical Labels for Top Tab Filing' for $103.60).
- Technology and Accessories sub-category generated revenue from two transactions: 3 units of 'Maxell 4.7GB DVD+R 5/Pack' ($2.97 total) and 1 unit of 'Apple iPhone 5S' ($569.99 total). However, the latter is a prominent transaction as it's

Due to the retrieve_data and generate_rag_response functions limiting the search to the top 3 chunks and due to the restrictions it was given in the guidelines it cannot give an accurate enough reponse with so little data.
<br/>
Changing n_results to a higher number makes the response generation time unbearably long

In [23]:
query_4 = "Which months show the highest sales? Is there seasonality?"
answer_4 = generate_rag_response(query_4)
print(f"Q: {query_4}\nAnswer: {answer_4}")


Generating response (Llama 3.2)...

Q: Which months show the highest sales? Is there seasonality?
Answer: To analyze the data and identify any seasonal patterns or the highest sales months, I'll calculate the total sales by month:

* December: $244.95 (Alice McCarthy's purchases)
	+ 6 * $9.02 = $54.12
	+ 2 * $69.46 = $138.92
	+ 2 * $10.86 = $21.72
	+ 3 * $79.47 = $237.41
* August: $462.56 (Edward Hooks' purchase)
* July:
	+ Scot Coram's purchases: $863.64 + $47.62 = $911.26
* June:
	+ Sarah Bern's purchases: $19.04 + $13.13 + $64.14 = $96.31
* May, March, April, October, November, December (2015 and 2017): No sales data available in this dataset

Based on the data, there seems to be seasonality:

* The highest monthly total sales are in July, with a combined total of $911.26.
* December is also one of the highest months, with a total of $244.95.

It's worth noting that the sample size for each month is relatively small due to the limited number of transactions provided. Therefore, whil

In [39]:
query_5 = "Which products are frequently sold at a discount?"
answer_5 = generate_rag_response(query_5)
print(f"Q: {query_5}\nAnswer: {answer_5}")

Generating response (Llama 3.2)...

Q: Which products are frequently sold at a discount?
Answer: Based on the provided transaction records, I can identify that the following product categories and individual products have been sold at a discount:

1. Bevis Boat-Shaped Conference Table (Furniture and Tables sub-category) - sold with a 0.3 discount on 2016-02-14
2. Wirebound Message Books, Four 2 3/4 x 5 Forms per Page, 200 Sets per Book (Office Supplies and Paper sub-category) - sold with a 0.0 discount but another instance of the same product was sold with a 0.2 discount on 2016-02-14 by Dave Kipp, however since the price changed, I am assuming that might be an error in the data.
3. GBC Premium Transparent Covers with Diagonal Lined Pattern (Office Supplies and Binders sub-category) - sold with a 0.2 discount on 2017-12-20


In [38]:
query_6 = "Which region has the best sales performance?"
answer_6 = generate_rag_response(query_6)
print(f"Q: {query_6}\nAnswer: {answer_6}")

Generating response (Llama 3.2)...

Q: Which region has the best sales performance?
Answer: Based on the provided transaction records, I can see that the South region has the highest sales performance. This is because they have purchased products in higher quantities and with higher profit margins compared to other regions.

In particular, Henry Goldwyn from Florence, Kentucky in the South region has made two purchases that stand out:

* 3 unit(s) of 'Quartet Omega Colored Chalk' for a total of $17.52
* 12 unit(s) of 'Premium Writing Pencils' for a total of $35.76

Additionally, Bart Watters from New York City, New York in the East region has also made two purchases that are worth noting:

* 3 unit(s) of 'Howard Miller Distant Time Traveler Alarm Clock' for a total of $82.26
* 5 unit(s) of 'Avery White Multi-Purpose Labels' for a total of $24.90

However, the South region's sales performance is still more impressive due to its higher volume and profit margins.

It's worth noting that t

In [37]:
query_7 = "How does the West region compare to the East in terms of profit?"
answer_7 = generate_rag_response(query_7, region_filter='East, West')
print(f"Q: {query_7}\nAnswer: {answer_7}")

Generating response (Llama 3.2)...

Q: How does the West region compare to the East in terms of profit?
Answer: To compare the profits between the West and East regions, I'll analyze the transaction data. However, please note that there are only two transactions mentioned for each region.

For the West region:
- Arianne Irving from Roseville, California purchased 3 units of 'Xerox 223' for a total of $19.44, generating a profit of $9.33 with a 0.0 discount.
- Denny Joy from San Francisco, California purchased 2 units of 'Luxo Economy Swing Arm Lamp' for a total of $39.88, generating a profit of $11.17 with a 0.0 discount.
- Denny Joy from San Francisco, California purchased 3 units of 'DAX Natural Wood-Tone Poster Frame' for a total of $79.44, generating a profit of $28.60 with a 0.0 discount.

For the East region:
- Liz MacKendrick from New York City, New York in the East regionpurchased 9 unit(s) of 'Dixon Prang Watercolor Pencils, 10-Color Set with Brush' for a total of $38.34, gene